# Acoustic refinement of an area change

A tapered duct (a horn / nozzle) is a **composite**: it discretizes a smooth area profile into a chain of compact area changes, each followed by a short duct.
Its **mean flow is exact at any resolution** -- an area change is a state-function relation, so the chain reproduces the continuous profile with no truncation error, and refining the segment count $N$ changes nothing.

Its **acoustics are a different story.**
A single compact area change is a zero-length impedance discontinuity: it reflects strongly and carries no transit phase.
A *distributed* horn spreads that impedance change over a length comparable to the wavelength, so it reflects far less (a gradual, near-adiabatic match) and adds real propagation delay.
Neither is captured at low $N$ -- so the **scattering matrix genuinely converges with refinement**, and the compact ($N=1$) limit can be badly wrong.

This notebook measures that convergence and contrasts it with the resolution-independent mean flow.

In [ ]:
import os, sys
# add the repo root (the dir containing the `nefes` package) to sys.path
_root = os.getcwd()
while not os.path.isdir(os.path.join(_root, "nefes")) and _root != os.path.dirname(_root):
    _root = os.path.dirname(_root)
sys.path.insert(0, _root)

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from nefes.shell import Network
from nefes.elements import catalog as cat
from nefes.elements.composite import auto_refine, grid_refine
from nefes.thermo.configure import perfect_gas
from nefes.perturbation import perturbation_response
from nefes.plotting.theme import NEFES_TEMPLATE_NAME

CFG = perfect_gas(287.0, 1.4)
L, A_IN, A_OUT = 0.4, 4.0e-3, 2.0e-3   # a 2:1 converging horn, 0.4 m long
PT_IN, P_OUT, T0 = 1.08e5, 1.0e5, 300.0
C0 = float(np.sqrt(1.4 * 287.0 * T0))  # ~347 m/s


def area(x):
    """Linear area profile A(x) from the inlet to the outlet."""
    return A_IN + (A_OUT - A_IN) * (x / L)


def horn(n):
    """The horn as a Network: inlet -> tapered_duct(N segments) -> outlet."""
    return Network(
        gas=CFG,
        nodes=[
            cat.total_pressure_inlet(PT_IN, T0),
            cat.tapered_duct(area, length=L, n_segments=n, name="horn"),
            cat.pressure_outlet(P_OUT, T0),
        ],
        edges=[(0, 1, A_IN), (1, 2, A_OUT)],
    )

## The mean flow is resolution-independent

Solve the horn at a coarse and a fine resolution: the exit Mach is identical to machine precision.
The lumped area changes are exact, so refinement adds interior sample points but no accuracy.

In [ ]:
for n in (1, 2, 8, 32):
    sol = horn(n).solve()
    print(f"N = {n:>3}   exit Mach = {sol.field('M')[1]:.8f}")

## The scattering matrix

The 2-port scattering matrix `S` maps the incoming acoustic waves (inlet + outlet) to the outgoing ones.
We read it between the **inlet edge (0)** and the **outlet edge (1)** across a frequency sweep, at several resolutions.
`|S11|` is the inlet reflection coefficient; watch it collapse as the horn is resolved -- and drop with frequency, as a longer horn (relative to the wavelength) becomes a better impedance match.

In [ ]:
def scatter(n, freqs):
    """Scattering matrix S(inlet edge 0 -> outlet edge 1), shape (n_freq, 2, 2)."""
    net = horn(n)
    sol = net.solve()
    prob = net.compile()
    resp = perturbation_response(prob, sol.x, np.atleast_1d(np.asarray(freqs, dtype=float)), excite=("acoustic",))
    return np.asarray(resp.scattering_matrix(0, 1)).reshape(-1, 2, 2)


freqs = np.linspace(40.0, 3000.0, 40)
resolutions = [1, 2, 4, 8, 32]
S = {n: scatter(n, freqs) for n in resolutions}

palette = ["#B0B0B0", "#4C78A8", "#72B7B2", "#E45756", "#111111"]
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=("inlet reflection  |S11|", "transmission phase  arg(S21)  [deg]"))
for col, n in zip(palette, resolutions):
    lbl = "N = 1 (compact)" if n == 1 else ("N = 32 (resolved)" if n == 32 else f"N = {n}")
    fig.add_scatter(x=freqs, y=np.abs(S[n][:, 0, 0]), name=lbl, line=dict(color=col), row=1, col=1)
    fig.add_scatter(x=freqs, y=np.degrees(np.angle(S[n][:, 1, 0])), name=lbl, line=dict(color=col),
                    showlegend=False, row=2, col=1)
fig.update_xaxes(title_text="frequency [Hz]", row=2, col=1)
fig.update_yaxes(title_text="|S11|", row=1, col=1)
fig.update_yaxes(title_text="arg(S21) [deg]", row=2, col=1)
fig.update_layout(template=NEFES_TEMPLATE_NAME, height=560,
                  title="Horn scattering matrix vs refinement (compact jump - resolved horn)")
fig

The compact model ($N=1$) is frequency-flat in reflection and badly overpredicts it, and its transmission phase misses the horn's transit delay.
As $N$ grows the reflection collapses (about an order of magnitude by a few kHz) and the phase settles -- the resolved, distributed horn.

## How fast does it converge?

At a fixed frequency, the distance of `S(N)` from a fine reference falls roughly like $\mathcal{O}(1/N)$ -- **first order**, far slower than the mean flow's instant convergence.
So an acoustic quantity needs genuine refinement (several segments per wavelength across the horn), not the one doubling a mean-flow quantity takes.

In [ ]:
f0 = 1500.0
S_ref = scatter(64, f0)[0]
Ns = np.array([1, 2, 4, 8, 16, 32])
err = np.array([np.linalg.norm(scatter(n, f0)[0] - S_ref) for n in Ns])

fig = go.Figure()
fig.add_scatter(x=Ns, y=err, mode="lines+markers", name="||S(N) - S_ref||", line=dict(color="#E45756", width=3))
fig.add_scatter(x=Ns, y=err[0] / Ns, mode="lines", name="O(1/N) guide",
                line=dict(color="gray", dash="dot"))
fig.update_xaxes(title_text="segments N", type="log")
fig.update_yaxes(title_text=f"scattering-matrix error at {f0:.0f} Hz", type="log")
fig.update_layout(template=NEFES_TEMPLATE_NAME, title="Scattering matrix converges ~ first order in N")
fig

## Converging an acoustic quantity with the refinement tools

`grid_refine` / `auto_refine` (the discretization-convergence helpers) work on **any** probed quantity -- including an acoustic one.
Point them at the reflection magnitude at the top frequency of interest and they refine until it settles.
The contrast with the mean flow is the whole story: the exit Mach converges in **one** doubling, the scattering matrix takes **several**.

A caveat worth stating: `|S11|` has a shallow plateau at very coarse $N$ before it drops, so a *loose* tolerance can "converge" prematurely on the plateau -- converge an acoustic QoI at the frequency and tolerance you actually care about.

In [ ]:
# mean-flow QoI: exit Mach -> converges in one doubling
mean_ref = grid_refine(lambda n: horn(n).solve(), 1, lambda s: {"M_exit": float(s.field("M")[1])})
print(f"mean flow  (exit Mach):  worst rel-change 1->2 = {mean_ref.worst:.2e}  -> converged at N=2")

# acoustic QoI: |S11| at 1500 Hz -> needs real refinement
def refl(n):
    return {"absS11": float(np.abs(scatter(n, f0)[0][0, 0]))}

ar = auto_refine(refl, 1, lambda d: d, tol=2e-2, max_refine=10)
print(f"acoustics  (|S11| @ {f0:.0f} Hz):  converged={ar.converged}  "
      f"at N={ar.n_final} after {ar.n_refine} refinements  ->  |S11| = {ar.final['absS11']:.4f}")

## Why

Acoustically, a lumped area change is a **zero-length impedance step**: strong reflection, no phase.
A distributed horn spreads that step over a length $\sim\lambda$, so it (a) reflects far less -- a gradual, near-adiabatic match, which is why `|S11|` also falls with frequency for the resolved horn -- and (b) adds genuine transit delay.
Refinement is what turns the staircase of compact steps into that continuous horn; the mean flow never needed it because an area change is exact lumped, but the acoustics do.

## Export for FNetLibUI

`network` and `sol` are the top-level objects the UI bridge reads (the resolved horn).
A composite element cannot yet be written to the UI YAML format, so the file export is skipped here -- build the network from atomic `isentropic_area_change` / `duct` primitives if you need the case file.

In [ ]:
import tempfile

network = horn(32)
sol = network.solve()
try:
    _out = os.path.join(tempfile.mkdtemp(), "acoustic_refinement_horn.yaml")
    sol.to_yaml(_out)  # embeds the mean-flow results; network.to_yaml(_out) for topology only
    print("saved case:", _out)
except ValueError as exc:
    print("YAML export skipped (composite element):", exc)